# 412 Study Data Cleaning

Cleans manual scoring CSVs:
- Drops `food type` column
- For each person with duplicate rows (one per food type):
  - If identical: keep one
  - If one row is all-zero and the other is not: keep the non-zero row
  - If both rows are all-zero: remove (degenerate)
  - If they genuinely differ: keep the food type 0 row
- Removes remaining degenerate rows (all scores = 0)
- For `total_donations` (no food type col): just removes degenerate rows

In [1]:
import pandas as pd
import os
import glob

In [2]:
def clean_manual_scoring_csv(filepath):
    df = pd.read_csv(filepath)
    filename = os.path.basename(filepath)
    print(f"\n=== {filename} ===")
    print(f"Original shape: {df.shape}")

    if 'food type' not in df.columns:
        # No food type column -- just remove degenerate rows
        score_cols = [c for c in df.columns if c != 'person']
        degenerate = (df[score_cols].apply(pd.to_numeric, errors='coerce') == 0).all(axis=1)
        removed = df[degenerate]['person'].tolist()
        df_clean = df[~degenerate].reset_index(drop=True)
        if removed:
            print(f"Removed degenerate persons: {removed}")
        print(f"Cleaned shape: {df_clean.shape}")
        return df_clean

    score_cols = [c for c in df.columns if c not in ('person', 'food type')]
    result_rows = []

    # Preserve original person order
    persons_seen = list(dict.fromkeys(df['person']))

    for person in persons_seen:
        group = df[df['person'] == person]

        if len(group) == 1:
            row = group.iloc[0]
            scores = pd.to_numeric(row[score_cols], errors='coerce').fillna(0)
            if (scores == 0).all():
                print(f"  Removed degenerate (single row, all zero): {person}")
                continue
            result_rows.append(row[['person'] + score_cols].to_dict())

        else:
            ft0_rows = group[group['food type'] == 0]
            ft1_rows = group[group['food type'] == 1]
            row0 = ft0_rows.iloc[0] if len(ft0_rows) > 0 else group.iloc[0]
            row1 = ft1_rows.iloc[0] if len(ft1_rows) > 0 else group.iloc[1]

            scores0 = pd.to_numeric(row0[score_cols], errors='coerce').fillna(0)
            scores1 = pd.to_numeric(row1[score_cols], errors='coerce').fillna(0)
            all_zero0 = (scores0 == 0).all()
            all_zero1 = (scores1 == 0).all()

            if all_zero0 and all_zero1:
                print(f"  Removed degenerate (both rows all zero): {person}")
            elif all_zero0:
                print(f"  {person}: food_type=0 all zero, keeping food_type=1")
                result_rows.append(row1[['person'] + score_cols].to_dict())
            elif all_zero1:
                print(f"  {person}: food_type=1 all zero, keeping food_type=0")
                result_rows.append(row0[['person'] + score_cols].to_dict())
            else:
                if scores0.equals(scores1):
                    result_rows.append(row0[['person'] + score_cols].to_dict())
                else:
                    print(f"  {person}: rows differ by food type -- keeping food_type=0")
                    result_rows.append(row0[['person'] + score_cols].to_dict())

    df_clean = pd.DataFrame(result_rows, columns=['person'] + score_cols).reset_index(drop=True)
    print(f"Cleaned shape: {df_clean.shape}")
    return df_clean

In [3]:
csv_files = sorted(glob.glob('Manual scoring*.csv'))
print('Files to clean:')
for f in csv_files:
    print(f'  {f}')

Files to clean:
  Manual scoring model database_sheet.xlsx - access.csv
  Manual scoring model database_sheet.xlsx - distance.csv
  Manual scoring model database_sheet.xlsx - income.csv
  Manual scoring model database_sheet.xlsx - last_donation.csv
  Manual scoring model database_sheet.xlsx - organization_size.csv
  Manual scoring model database_sheet.xlsx - poverty.csv
  Manual scoring model database_sheet.xlsx - total_donations_all types.csv


In [4]:
cleaned = {}
for filepath in csv_files:
    df_clean = clean_manual_scoring_csv(filepath)
    cleaned[filepath] = df_clean
    df_clean.to_csv(filepath, index=False)
    print(f'  Saved: {filepath}')


=== Manual scoring model database_sheet.xlsx - access.csv ===
Original shape: (15, 4)
Cleaned shape: (15, 4)
  Saved: Manual scoring model database_sheet.xlsx - access.csv

=== Manual scoring model database_sheet.xlsx - distance.csv ===
Original shape: (13, 5)
Cleaned shape: (13, 5)
  Saved: Manual scoring model database_sheet.xlsx - distance.csv

=== Manual scoring model database_sheet.xlsx - income.csv ===
Original shape: (13, 7)
Cleaned shape: (13, 7)
  Saved: Manual scoring model database_sheet.xlsx - income.csv

=== Manual scoring model database_sheet.xlsx - last_donation.csv ===
Original shape: (14, 14)
Cleaned shape: (14, 14)
  Saved: Manual scoring model database_sheet.xlsx - last_donation.csv

=== Manual scoring model database_sheet.xlsx - organization_size.csv ===
Original shape: (16, 6)
Cleaned shape: (16, 6)
  Saved: Manual scoring model database_sheet.xlsx - organization_size.csv

=== Manual scoring model database_sheet.xlsx - poverty.csv ===
Original shape: (15, 8)
Clean

In [5]:
for filepath, df in cleaned.items():
    print(f"\n=== {os.path.basename(filepath)} ===")
    display(df)


=== Manual scoring model database_sheet.xlsx - access.csv ===


,person,extremely low,low,normal
0,D4,10,7.5,5
1,R2,10,5.0,3
2,R3,30,20.0,0
3,V1,60,60.0,0
4,V3,20,10.0,0
5,V5,30,20.0,10
6,V6,5,3.0,0
7,V7,5,4.0,1
8,R1,30,20.0,10
9,F3,30,15.0,0



=== Manual scoring model database_sheet.xlsx - distance.csv ===


,person,15,30,45,60+
0,D4,10,5,0,-10
1,R3,30,30,15,0
2,V1,50,50,30,30
3,V3,60,20,0,-20
4,V6,25,15,5,0
5,R1,30,30,15,15
6,F3,30,10,0,-5
7,R8,10,10,10,30
8,F2,30,0,0,-10
9,D2,4,4,4,1



=== Manual scoring model database_sheet.xlsx - income.csv ===


...

,person,<50,50-100,100-500,500-1000,1000
0,D4,5,5,10,10,5
1,R2,4,4,4,5,5
2,R3,0,0,15,30,30
3,V1,30,30,30,40,40
4,V3,10,20,20,10,5
5,V5,20,20,20,30,30
6,V6,10,10,12,12,15
7,V7,1,1,1,2,2
8,R1,20,20,25,30,15
9,F3,7,15,30,15,7



=== Manual scoring model database_sheet.xlsx - poverty.csv ===


,person,0 - 10%,10% - 20%,20% - 30%,30% - 40%,40% - 50%,50% - 60%,60% +
0,D4,1,1.0,5.0,5.0,10,10.0,10.0
1,R2,5,5.0,8.0,8.0,10,10.0,10.0
2,R3,0,15.0,15.0,30.0,30,30.0,30.0
3,V1,0,0.0,0.0,60.0,60,60.0,60.0
4,V3,0,0.0,10.0,20.0,25,30.0,40.0
5,V5,2,10.0,20.0,30.0,30,30.0,40.0
6,V6,0,0.0,0.0,3.0,3,5.0,5.0
7,V7,0,0.5,1.0,3.0,4,5.0,5.5
8,R1,0,15.0,20.0,20.0,20,30.0,30.0
9,F3,0,0.0,15.0,15.0,30,30.0,30.0



=== Manual scoring model database_sheet.xlsx - total_donations_all types.csv ===


,person,0,1,2,3,4,5,6,7,8,9,10,11,12
0,R3,0,0.0,0,0.0,0,30.0,30,30.0,30,30.0,30,30.0,30
1,V1,60,60.0,60,60.0,60,60.0,60,60.0,60,60.0,60,30.0,30
2,V3,0,12.0,11,10.0,9,8.0,7,6.0,5,4.0,3,2.0,1
3,V5,28,28.0,10,10.0,10,10.0,10,10.0,10,10.0,2,2.0,2
4,V6,10,25.0,25,25.0,25,18.0,18,18.0,18,12.0,12,12.0,12
5,V7,2,2.0,2,2.0,2,2.0,2,1.0,1,1.0,1,1.0,1
6,R1,30,30.0,30,30.0,20,20.0,20,15.0,15,15.0,10,10.0,10
7,F3,30,30.0,30,30.0,30,15.0,15,15.0,15,0.0,0,0.0,0
8,R8,60,10.0,10,10.0,10,10.0,10,10.0,10,10.0,10,10.0,60
9,F2,15,15.0,15,15.0,15,12.0,12,12.0,12,12.0,12,10.0,10
